# 14 — Laser Weld Defect Detection · Co-Training (Semi-Supervised)
## LozanoLsa | Operational Excellence · MBB · Machine Learning
---

### Two machines. One shared question.

On a laser welding line for automotive body panels, two independent data streams
run simultaneously — always, every weld, every shift:

**View A** captures the *process parameters* set by the operator and the machine
controller: energy input, travel speed, pulse frequency, spot diameter, shielding
gas flow. These are the conditions under which the weld was made.

**View B** captures the *response of the material*: bead height measured by laser
profilometry, thermal variance from the pyrometer, acoustic emission RMS, penetration
depth from ultrasound, and vibration signature during deposition. These are the
consequences of how the process unfolded.

Both views describe the same weld. But they describe it from entirely different
physical perspectives — and that independence is the key to **Co-Training**.

The problem: of **1,500 welds** logged during production, only **150 were physically
inspected** (cross-sectioned, radiographed, or tensile-tested). The rest exist only
as sensor streams with no confirmed quality verdict. Physical inspection is destructive,
slow, and expensive. It cannot scale to full-volume production.

Co-Training exploits the two independent views to teach each model from what the
other is most confident about — iteratively, across multiple rounds, using
the unlabeled 90% of production data as its expansion fuel.

> ⚠️ **Deployment note:** Co-Training probabilities are inputs to a quality alert
> workflow — not autonomous reject decisions. Final disposition of flagged welds
> must be confirmed by a quality technician. The model's role is triage, not judgment.


---
## 2 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    recall_score, precision_score, confusion_matrix,
    roc_curve
)
from sklearn.calibration import calibration_curve

RANDOM_STATE = 42    # used only for train/test split
THRESHOLD    = 0.65  # confidence threshold for pseudo-labeling
ROUNDS       = 5     # Co-Training iterations

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
CYAN = '#4C9BE8'
RED  = '#E8574C'
GRAY = '#3a3f4b'


---
## 3 · Load Data

### Column reference — two independent views

**View A — Process Parameters (set by machine controller)**

| Column | Units | Description |
|--------|-------|-------------|
| `energy_j` | J | Laser pulse energy |
| `speed_mm_s` | mm/s | Welding travel speed |
| `frequency_khz` | kHz | Pulse repetition frequency |
| `spot_diameter_mm` | mm | Focused beam spot diameter |
| `gas_flow_l_min` | L/min | Shielding gas flow rate |

**View B — In-Process Sensor Monitoring (material response)**

| Column | Units | Description |
|--------|-------|-------------|
| `bead_height_mm` | mm | Weld bead height (laser profilometry) |
| `thermal_var_c` | °C | Pyrometer thermal variance |
| `acoustic_rms` | — | Acoustic emission RMS signal |
| `penetration_mm` | mm | Weld penetration depth |
| `vibration_rms` | — | Vibration RMS during deposition |

**Target & metadata**

| Column | Values | Description |
|--------|--------|-------------|
| `is_labeled` | 0/1 | 1 = physically inspected |
| `defect_label` | 0/1/NaN | Quality verdict (labeled only) |
| `defect_real` | 0/1 | Ground truth (analysis only — not available in production) |


In [ ]:
try:
    df = pd.read_csv("welding_data.csv")
except FileNotFoundError:
    base = "https://raw.githubusercontent.com/LozanoLsa/CoTraining_Welding/main/"
    df = pd.read_csv(base + "welding_data.csv")

print(f"Total welds   : {len(df):,}")
print(f"Labeled (inspected): {df['is_labeled'].sum()} ({df['is_labeled'].mean()*100:.0f}%)")
print(f"Unlabeled     : {(df['is_labeled']==0).sum()} ({(1-df['is_labeled'].mean())*100:.0f}%)")
df.head(8)


In [ ]:
df_lab = df[df["is_labeled"]==1]
print("Among labeled welds:")
vc = df_lab["defect_label"].value_counts()
print(f"  Conforming (0): {int(vc.get(0.0,0))} ({int(vc.get(0.0,0))/len(df_lab)*100:.0f}%)")
print(f"  Defective  (1): {int(vc.get(1.0,0))} ({int(vc.get(1.0,0))/len(df_lab)*100:.0f}%)")
print(f"  Global defect rate (y_real): {df['defect_real'].mean()*100:.1f}%")


---
## 4 · Sanity Checks

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\n(NaN in defect_label = unlabeled welds — expected)")


In [ ]:
print("Data types:")
print(df.dtypes)
print("\nDescriptive stats:")
print(df.describe().round(3))


In [ ]:
assert df["energy_j"].between(1, 10).all(), "Energy out of expected range"
assert df["spot_diameter_mm"].between(0.3, 2.0).all(), "Spot diameter anomaly"
assert df["is_labeled"].isin([0,1]).all(), "Unexpected is_labeled values"
print(f"Duplicates: {df.duplicated().sum()}")
print("All domain constraints validated.")


---
## 5 · Exploratory Data Analysis

Three EDA lenses tailored to the Co-Training setup:

1. **Are the two views genuinely independent?** — Correlation between View A and B features.
   If they were highly correlated, the mutual teaching mechanism would be redundant.
2. **What physical conditions associate with defects?** — Within the labeled 10%.
3. **How separable are defects by thermal variance?** — The sensor expected to carry most signal.


In [ ]:
# EDA 1 — Cross-view feature correlation
FEATURES_A = ["energy_j","speed_mm_s","frequency_khz","spot_diameter_mm","gas_flow_l_min"]
FEATURES_B = ["bead_height_mm","thermal_var_c","acoustic_rms","penetration_mm","vibration_rms"]

corr_cross = df[FEATURES_A + FEATURES_B].corr().loc[FEATURES_A, FEATURES_B]

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(corr_cross, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.4, ax=ax, vmin=-1, vmax=1)
ax.set_title("Cross-View Feature Correlation (View A vs View B)",
             fontweight="bold", fontsize=12)
ax.set_xlabel("View B — Sensor Response")
ax.set_ylabel("View A — Process Parameters")
plt.tight_layout()
plt.show()

max_corr = corr_cross.abs().values.max()
print(f"Max absolute cross-view correlation: {max_corr:.3f}")
print("Low cross-view correlation = views are sufficiently independent for Co-Training.")


In [ ]:
# EDA 2 — Defect rate by energy quartile (labeled set)
df_lab = df[df["is_labeled"]==1].copy()
df_lab["energy_q"] = pd.qcut(df_lab["energy_j"], q=4,
                               labels=["Q1\n(low)","Q2","Q3","Q4\n(high)"])

defect_by_q = df_lab.groupby("energy_q")["defect_label"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(defect_by_q.index, defect_by_q.values,
              color=[RED if v > 0.25 else CYAN for v in defect_by_q.values],
              edgecolor="white", linewidth=0.5)
ax.set_ylabel("Defect Rate")
ax.set_xlabel("Energy Quartile")
ax.set_title("Defect Rate by Laser Energy Quartile (Labeled Set)", fontweight="bold")
ax.set_ylim(0, 0.6)
for bar, val in zip(bars, defect_by_q.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.0%}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# EDA 3 — Thermal variance distribution: conforming vs defective
conf    = df_lab[df_lab["defect_label"]==0]["thermal_var_c"]
defect  = df_lab[df_lab["defect_label"]==1]["thermal_var_c"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(conf,   bins=20, alpha=0.75, color=CYAN, label="Conforming")
ax.hist(defect, bins=20, alpha=0.70, color=RED,  label="Defective")
ax.axvline(conf.mean(),   color=CYAN, linestyle="--", lw=1.8,
           label=f"Conforming mean: {conf.mean():.1f}°C")
ax.axvline(defect.mean(), color=RED,  linestyle="--", lw=1.8,
           label=f"Defective mean:  {defect.mean():.1f}°C")
ax.set_xlabel("Thermal Variance (°C)")
ax.set_ylabel("Weld Count")
ax.set_title("Thermal Variance Distribution by Quality Class", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Conforming mean thermal variance : {conf.mean():.2f}°C")
print(f"Defective mean thermal variance  : {defect.mean():.2f}°C")
print(f"Separation delta                 : {defect.mean()-conf.mean():.2f}°C")


---
## 6 · Preprocessing

### Two separate scalers — one per view

Co-Training requires each view to be processed independently. StandardScaler
is embedded inside each Pipeline to prevent any information leakage between
View A and View B during training or pseudo-labeling.

**Why `class_weight='balanced'`?**
With only 150 labeled samples and a 30% defect rate in that subset, unweighted
Logistic Regression tends to collapse to all-negative predictions after the
Co-Training expansion dilutes the defect signal. Balanced weighting corrects
the per-class loss function and keeps the model sensitive to the minority class.

**Why threshold 0.65 instead of 0.80?**
Logistic Regression produces well-calibrated probabilities but has less
overconfidence than tree ensembles. A threshold of 0.65 captures enough
confident pseudo-labels per round to produce meaningful dataset growth
without flooding the training pool with borderline predictions.


In [ ]:
FEATURES_A = ["energy_j","speed_mm_s","frequency_khz","spot_diameter_mm","gas_flow_l_min"]
FEATURES_B = ["bead_height_mm","thermal_var_c","acoustic_rms","penetration_mm","vibration_rms"]
TARGET     = "defect_label"

df_labeled   = df[df["is_labeled"]==1].copy()
df_unlabeled = df[df["is_labeled"]==0].copy()

X_A_lab = df_labeled[FEATURES_A]
X_B_lab = df_labeled[FEATURES_B]
y_lab   = df_labeled[TARGET].astype(int)

# Stratified split — RANDOM_STATE used only here
X_A_tr, X_A_te, y_tr, y_te = train_test_split(
    X_A_lab, y_lab, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y_lab
)
X_B_tr = df_labeled.loc[X_A_tr.index, FEATURES_B]
X_B_te = df_labeled.loc[X_A_te.index, FEATURES_B]
idx_tr = X_A_tr.index.to_numpy()

print(f"Training pool : {len(X_A_tr)} labeled welds")
print(f"Test set      : {len(X_A_te)} labeled welds (held-out)")
print(f"Unlabeled pool: {len(df_unlabeled)} welds")
print(f"\nDefect rate in training: {y_tr.mean()*100:.1f}%")


In [ ]:
# Base learners — one per view, identical architecture
base_A = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=500, class_weight="balanced"))
])

base_B = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=500, class_weight="balanced"))
])

print("Pipelines configured. Each view has its own independent scaler + classifier.")


---
## 7 · Co-Training Loop

### The mechanism in plain language

```
VIEW A (process params)          VIEW B (sensor response)
  ┌───────────────┐                ┌───────────────┐
  │ Model A       │                │ Model B       │
  │ (LogReg + SS) │                │ (LogReg + SS) │
  └──────┬────────┘                └──────┬────────┘
         │ scores unlabeled               │ scores unlabeled
         │                               │
   P_A(defect) ≥ 0.65 → donate to B   P_B(defect) ≥ 0.65 → donate to A
   P_A(defect) ≤ 0.35 → donate to B   P_B(defect) ≤ 0.35 → donate to A
         │                               │
         └──────────► ROUND N+1 ◄────────┘
```

Key property: Model A only teaches Model B, and vice versa.
Neither model teaches itself — that is what separates Co-Training from Self-Training.
The cross-view constraint enforces diversity and reduces error propagation.


In [ ]:
global_unlab = df_unlabeled.index.to_numpy()

# Initialize with the labeled training set
cur_A_idx = idx_tr.copy(); cur_A_y = y_tr.values.copy()
cur_B_idx = idx_tr.copy(); cur_B_y = y_tr.values.copy()

hist = {"A_acc":[],"A_f1":[],"A_auc":[],"A_recall":[],
        "B_acc":[],"B_f1":[],"B_auc":[],"B_recall":[],
        "A_size":[],"B_size":[]}

for r in range(ROUNDS):
    # 1 — Clone and retrain both models with current labeled pools
    mA = clone(base_A); mB = clone(base_B)
    mA.fit(df.loc[cur_A_idx, FEATURES_A], cur_A_y)
    mB.fit(df.loc[cur_B_idx, FEATURES_B], cur_B_y)

    # 2 — Evaluate on all 1,500 welds using y_real (analysis-only ground truth)
    y_all = df["defect_real"].values
    pA_all = mA.predict(df[FEATURES_A]); prA_all = mA.predict_proba(df[FEATURES_A])[:,1]
    pB_all = mB.predict(df[FEATURES_B]); prB_all = mB.predict_proba(df[FEATURES_B])[:,1]

    hist["A_acc"].append(round(accuracy_score(y_all, pA_all),4))
    hist["A_f1"].append(round(f1_score(y_all, pA_all, zero_division=0),4))
    hist["A_auc"].append(round(roc_auc_score(y_all, prA_all),4))
    hist["A_recall"].append(round(recall_score(y_all, pA_all, zero_division=0),4))
    hist["A_size"].append(len(cur_A_idx))

    hist["B_acc"].append(round(accuracy_score(y_all, pB_all),4))
    hist["B_f1"].append(round(f1_score(y_all, pB_all, zero_division=0),4))
    hist["B_auc"].append(round(roc_auc_score(y_all, prB_all),4))
    hist["B_recall"].append(round(recall_score(y_all, pB_all, zero_division=0),4))
    hist["B_size"].append(len(cur_B_idx))

    print(f"Round {r+1}: A [F1={hist['A_f1'][-1]:.4f} AUC={hist['A_auc'][-1]:.4f} "
          f"n={hist['A_size'][-1]}]  B [F1={hist['B_f1'][-1]:.4f} "
          f"AUC={hist['B_auc'][-1]:.4f} n={hist['B_size'][-1]}]")

    # 3 — Score the unlabeled pool
    pr_A_u = mA.predict_proba(df.loc[global_unlab, FEATURES_A])[:,1]
    pr_B_u = mB.predict_proba(df.loc[global_unlab, FEATURES_B])[:,1]

    # 4 — Select confident predictions (each view donates to the OTHER)
    iA1=np.where(pr_A_u>THRESHOLD)[0]; iA0=np.where(pr_A_u<(1-THRESHOLD))[0]
    iB1=np.where(pr_B_u>THRESHOLD)[0]; iB0=np.where(pr_B_u<(1-THRESHOLD))[0]

    A_ci=np.concatenate([global_unlab[iA1],global_unlab[iA0]])
    A_cy=np.concatenate([np.ones(len(iA1)), np.zeros(len(iA0))])
    B_ci=np.concatenate([global_unlab[iB1],global_unlab[iB0]])
    B_cy=np.concatenate([np.ones(len(iB1)), np.zeros(len(iB0))])

    # 5 — Expand pools (B teaches A, A teaches B)
    cur_A_idx=np.concatenate([cur_A_idx, B_ci]); cur_A_y=np.concatenate([cur_A_y, B_cy])
    cur_B_idx=np.concatenate([cur_B_idx, A_ci]); cur_B_y=np.concatenate([cur_B_y, A_cy])

    # 6 — Deduplicate (keep first label assigned)
    _, ui=np.unique(cur_A_idx,return_index=True); cur_A_idx=cur_A_idx[ui]; cur_A_y=cur_A_y[ui]
    _, ui=np.unique(cur_B_idx,return_index=True); cur_B_idx=cur_B_idx[ui]; cur_B_y=cur_B_y[ui]


In [ ]:
# Convergence visualization
rounds = list(range(1, ROUNDS+1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(rounds, hist["A_f1"], "o-", color=CYAN,  linewidth=2, label="Model A (Process)")
axes[0].plot(rounds, hist["B_f1"], "o-", color=RED,   linewidth=2, label="Model B (Sensors)")
axes[0].set_xlabel("Co-Training Round"); axes[0].set_ylabel("F1 Score")
axes[0].set_title("F1 Score Convergence", fontweight="bold")
axes[0].legend(); axes[0].set_xticks(rounds)

axes[1].plot(rounds, hist["A_auc"], "o-", color=CYAN, linewidth=2, label="Model A (Process)")
axes[1].plot(rounds, hist["B_auc"], "o-", color=RED,  linewidth=2, label="Model B (Sensors)")
axes[1].set_xlabel("Co-Training Round"); axes[1].set_ylabel("AUC-ROC")
axes[1].set_title("AUC-ROC Convergence", fontweight="bold")
axes[1].legend(); axes[1].set_xticks(rounds)

plt.tight_layout()
plt.show()


---
## 8 · Evaluation

Final models are retrained on the full accumulated labeled pools.
Evaluation uses the **held-out test set (real labels only)** — never pseudo-labels.


In [ ]:
# Train final models on full accumulated pools
model_A_final = clone(base_A)
model_B_final = clone(base_B)

model_A_final.fit(df.loc[cur_A_idx, FEATURES_A], cur_A_y)
model_B_final.fit(df.loc[cur_B_idx, FEATURES_B], cur_B_y)

print(f"Final Model A trained on {len(cur_A_idx)} samples (real + pseudo)")
print(f"Final Model B trained on {len(cur_B_idx)} samples (real + pseudo)")


In [ ]:
# Held-out test evaluation
yp_A  = model_A_final.predict(X_A_te)
yp_B  = model_B_final.predict(X_B_te)
pr_A  = model_A_final.predict_proba(X_A_te)[:,1]
pr_B  = model_B_final.predict_proba(X_B_te)[:,1]
pr_ens = (pr_A + pr_B) / 2.0
yp_ens = (pr_ens >= 0.50).astype(int)

results = {}
for name, yp, pr in [("Model A (Process)", yp_A, pr_A),
                      ("Model B (Sensors)", yp_B, pr_B),
                      ("Ensemble (avg)",    yp_ens, pr_ens)]:
    results[name] = {
        "Accuracy" : round(accuracy_score(y_te, yp), 4),
        "AUC-ROC"  : round(roc_auc_score(y_te, pr), 4),
        "F1"       : round(f1_score(y_te, yp, zero_division=0), 4),
        "Recall"   : round(recall_score(y_te, yp, zero_division=0), 4),
        "Precision": round(precision_score(y_te, yp, zero_division=0), 4),
    }

df_results = pd.DataFrame(results).T
print("=== Test Set Metrics ===\n")
print(df_results.to_string())


In [ ]:
# Confusion matrices — side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, yp) in zip(axes, [("Model A (Process)", yp_A),
                                   ("Model B (Sensors)", yp_B),
                                   ("Ensemble",          yp_ens)]):
    cm = confusion_matrix(y_te, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Predicted OK","Predicted Defect"],
                yticklabels=["Actual OK","Actual Defect"],
                linewidths=0.5, linecolor="white")
    ax.set_title(name, fontweight="bold")

plt.suptitle("Confusion Matrices — Held-Out Test Set", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves — all three on one chart
fig, ax = plt.subplots(figsize=(7, 5))

for name, pr, col in [("Model A (Process)", pr_A, CYAN),
                       ("Model B (Sensors)", pr_B, RED),
                       ("Ensemble",          pr_ens, "gold")]:
    auc = roc_auc_score(y_te, pr)
    fpr, tpr, _ = roc_curve(y_te, pr)
    ax.plot(fpr, tpr, linewidth=2.5, color=col, label=f"{name} (AUC={auc:.4f})")

ax.plot([0,1],[0,1],"k--",linewidth=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Co-Training Final Models", fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


---
## 9 · Interpretability

In [ ]:
# Logistic Regression coefficients (post-scaling = comparable across features)
coef_A = model_A_final.named_steps["clf"].coef_[0]
coef_B = model_B_final.named_steps["clf"].coef_[0]

df_coef_A = pd.DataFrame({"feature": FEATURES_A, "coefficient": coef_A})\
                .sort_values("coefficient", key=abs, ascending=False)
df_coef_B = pd.DataFrame({"feature": FEATURES_B, "coefficient": coef_B})\
                .sort_values("coefficient", key=abs, ascending=False)

print("Model A — Standardized Coefficients (Process View):")
print(df_coef_A.to_string(index=False))
print("\nModel B — Standardized Coefficients (Sensor View):")
print(df_coef_B.to_string(index=False))


In [ ]:
# Coefficient comparison charts
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, df_c, title, feats in [
    (axes[0], df_coef_A, "Model A — Process View", FEATURES_A),
    (axes[1], df_coef_B, "Model B — Sensor View",  FEATURES_B)
]:
    colors = [RED if c > 0 else CYAN for c in df_c["coefficient"]]
    ax.barh(df_c["feature"][::-1], df_c["coefficient"][::-1], color=colors[::-1])
    ax.axvline(0, color="white", linewidth=0.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Standardized Coefficient")

plt.suptitle("Co-Training Feature Coefficients\n(Red = increases defect risk, Blue = protective)",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Calibration curves — do probabilities mean what they say?
pr_A_all  = model_A_final.predict_proba(df[FEATURES_A])[:,1]
pr_B_all  = model_B_final.predict_proba(df[FEATURES_B])[:,1]
y_all_real = df["defect_real"].values

fp_A, mp_A = calibration_curve(y_all_real, pr_A_all, n_bins=8, strategy="quantile")
fp_B, mp_B = calibration_curve(y_all_real, pr_B_all, n_bins=8, strategy="quantile")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mp_A, fp_A, "o-", color=CYAN, linewidth=2, label="Model A (Process)")
ax.plot(mp_B, fp_B, "o-", color=RED,  linewidth=2, label="Model B (Sensors)")
ax.plot([0,1],[0,1],"k--", linewidth=1, label="Perfect calibration")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Defect Fraction")
ax.set_title("Calibration Curves (Reliability Diagram)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("A well-calibrated model: P=0.70 means ~70% of those welds are actually defective.")
print("Deviation from the diagonal reveals systematic over- or under-confidence.")


---
## 10 · Semi-Supervised Validation

Three validation angles specific to Co-Training:

1. **Dataset growth trajectory** — how many pseudo-labels were added per round
2. **Pseudo-label quality check** — agreement rate between Model A and Model B
3. **View independence confirmation** — ensures the cross-teaching signal is genuine


In [ ]:
# Validation 1 — Dataset size growth per round
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rounds, hist["A_size"], "o-", color=CYAN, linewidth=2, label="Pool A")
ax.plot(rounds, hist["B_size"], "o-", color=RED,  linewidth=2, label="Pool B")
ax.axhline(len(idx_tr), color="gray", linestyle="--", linewidth=1, label="Initial labeled (120)")
ax.set_xlabel("Co-Training Round")
ax.set_ylabel("Training Pool Size")
ax.set_title("Training Pool Growth Across Co-Training Rounds", fontweight="bold")
ax.set_xticks(rounds)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Validation 2 — Pseudo-label agreement between the two models
yp_A_all = model_A_final.predict(df[FEATURES_A])
yp_B_all = model_B_final.predict(df[FEATURES_B])

agreement = (yp_A_all == yp_B_all).mean()
both_defect = ((yp_A_all==1) & (yp_B_all==1)).sum()
a_only      = ((yp_A_all==1) & (yp_B_all==0)).sum()
b_only      = ((yp_A_all==0) & (yp_B_all==1)).sum()

print(f"Overall prediction agreement: {agreement*100:.1f}%")
print(f"Both flag defect    : {both_defect}")
print(f"Only Model A flags  : {a_only}")
print(f"Only Model B flags  : {b_only}")
print()
print("Consensus defects (both agree) carry the highest operational confidence.")
print("Disagreements warrant priority physical inspection.")


In [ ]:
# Validation 3 — Recall improvement vs baseline (labeled-only)
base_only_A = clone(base_A)
base_only_B = clone(base_B)
base_only_A.fit(X_A_tr, y_tr)
base_only_B.fit(X_B_tr, y_tr)

recall_A_base = recall_score(y_te, base_only_A.predict(X_A_te), zero_division=0)
recall_B_base = recall_score(y_te, base_only_B.predict(X_B_te), zero_division=0)
recall_A_co   = recall_score(y_te, yp_A, zero_division=0)
recall_B_co   = recall_score(y_te, yp_B, zero_division=0)

print("Recall improvement from Co-Training:")
print(f"  Model A: {recall_A_base:.4f} → {recall_A_co:.4f}  (Δ = {recall_A_co - recall_A_base:+.4f})")
print(f"  Model B: {recall_B_base:.4f} → {recall_B_co:.4f}  (Δ = {recall_B_co - recall_B_base:+.4f})")


---
## 11 · Simulator & Final Reflection

In [ ]:
def simulate_weld_defect(
    energy_j, speed_mm_s, frequency_khz, spot_diameter_mm, gas_flow_l_min,
    bead_height_mm, thermal_var_c, acoustic_rms, penetration_mm, vibration_rms
):
    """
    Estimate weld defect probability using the Co-Training ensemble.
    Returns individual model probabilities and the ensemble average.

    Risk thresholds (ensemble P):
    - P ≥ 0.70 : HIGH — immediate inspection recommended
    - P ≥ 0.45 : MODERATE — flag for next quality audit cycle
    - P < 0.45 : LOW — weld within expected conformance range
    """
    row_A = pd.DataFrame([{
        "energy_j": energy_j, "speed_mm_s": speed_mm_s,
        "frequency_khz": frequency_khz, "spot_diameter_mm": spot_diameter_mm,
        "gas_flow_l_min": gas_flow_l_min
    }])
    row_B = pd.DataFrame([{
        "bead_height_mm": bead_height_mm, "thermal_var_c": thermal_var_c,
        "acoustic_rms": acoustic_rms, "penetration_mm": penetration_mm,
        "vibration_rms": vibration_rms
    }])

    p_A = model_A_final.predict_proba(row_A)[0, 1]
    p_B = model_B_final.predict_proba(row_B)[0, 1]
    p_ens = (p_A + p_B) / 2.0

    if p_ens >= 0.70:
        risk = "🔴 HIGH — Inspection recommended"
    elif p_ens >= 0.45:
        risk = "🟡 MODERATE — Monitor closely"
    else:
        risk = "🟢 LOW — Within conformance range"

    print(f"P(Defect) — Model A (Process): {p_A:.3f}")
    print(f"P(Defect) — Model B (Sensors): {p_B:.3f}")
    print(f"P(Defect) — Ensemble average : {p_ens:.3f}   →   {risk}")
    return round(p_A,3), round(p_B,3), round(p_ens,3)


In [ ]:
# Scenario A — Defective weld signature (low energy, high thermal variance)
print("=== Scenario A: High-Risk Weld ===")
simulate_weld_defect(
    energy_j=3.2, speed_mm_s=23.0, frequency_khz=7.5,
    spot_diameter_mm=0.75, gas_flow_l_min=11.0,
    bead_height_mm=1.05, thermal_var_c=28.0,
    acoustic_rms=0.62, penetration_mm=1.8, vibration_rms=0.32
)


In [ ]:
# Scenario B — Conforming weld (optimal parameters)
print("=== Scenario B: Conforming Weld ===")
simulate_weld_defect(
    energy_j=4.1, speed_mm_s=20.0, frequency_khz=8.0,
    spot_diameter_mm=0.80, gas_flow_l_min=12.0,
    bead_height_mm=1.22, thermal_var_c=20.0,
    acoustic_rms=0.60, penetration_mm=2.1, vibration_rms=0.30
)


In [ ]:
# Scenario C — Same as A but energy corrected to nominal
print("=== Scenario C: Scenario A with corrected energy (4.0 J) ===")
simulate_weld_defect(
    energy_j=4.0, speed_mm_s=23.0, frequency_khz=7.5,
    spot_diameter_mm=0.75, gas_flow_l_min=11.0,
    bead_height_mm=1.12, thermal_var_c=24.0,
    acoustic_rms=0.61, penetration_mm=1.9, vibration_rms=0.31
)
print("\nInsight: restoring energy from 3.2 J to nominal 4.0 J substantially")
print("Co-Training reduced defect probability - process control matters.")


In [ ]:
# Grid: defect probability vs energy and thermal variance
energy_vals = np.arange(3.0, 5.1, 0.25)
thermo_vals = [18, 22, 26, 30]

grid = []
for e in energy_vals:
    for t in thermo_vals:
        pA,pB,pe = simulate_weld_defect(
            energy_j=e, speed_mm_s=20.0, frequency_khz=8.0,
            spot_diameter_mm=0.80, gas_flow_l_min=12.0,
            bead_height_mm=1.2, thermal_var_c=t,
            acoustic_rms=0.60, penetration_mm=2.0, vibration_rms=0.30
        )
        grid.append({"energy_j": round(e,2), "thermal_var_c": t, "p_ensemble": pe})

df_grid = pd.DataFrame(grid).pivot(index="thermal_var_c", columns="energy_j", values="p_ensemble")

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df_grid, annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=ax, vmin=0, vmax=1, cbar_kws={"label": "P(Defect) Ensemble"})
ax.set_title("Defect Probability: Energy × Thermal Variance\n(Reference: speed=20 mm/s, all other params nominal)", fontweight="bold")
ax.set_xlabel("Laser Energy (J)")
ax.set_ylabel("Thermal Variance (°C)")
plt.tight_layout()
plt.show()


---
### Final Reflection

Co-Training rests on a single bet: that two genuinely independent descriptions
of the same phenomenon will agree more often when they are right than when they
are wrong. If that bet holds, their disagreements become informative rather than
just noisy — they mark the cases most worth inspecting.

On this laser welding line, the bet held. The process view (View A) and the sensor
view (View B) emerged from different physical measurements and, as the correlation
analysis showed, from genuinely different information spaces. That independence
gave the mutual teaching mechanism a real signal to propagate.

The honest limits are worth naming: the labeled pool was small and the defect rate
was moderate. On an actual production line, the threshold of 0.65 would need
empirical calibration against real inspection outcomes — not a fixed design choice.
And the "analysis-only" ground truth column (`defect_real`) would not exist;
only the welds that get physically inspected would ever reveal their true nature.

That last point is perhaps the most important one. In a real Co-Training deployment,
the model's pseudo-labels should feed a **selective inspection queue** — not bypass
inspection entirely. Each physically confirmed pseudo-label strengthens the next round.
The algorithm and the inspector become collaborators, not substitutes for each other.

---
*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*
